In [2]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import martini_openmm as martini

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

## Define simulation parameters

In [3]:
temperature = 298*unit.kelvin
timestep = 10*unit.femtosecond
pressure = 1*unit.bar

max_K = 25
K_gradient = 0.001

report_interval = 1000

## Set up Martini3 system

In [4]:
conf = GromacsGroFile(f"Data/2ND3/system.gro")
box_vectors = conf.getPeriodicBoxVectors()
topfile = martini.MartiniTopFile(
    f"Data/2ND3/system.top",
    periodicBoxVectors=box_vectors,
    epsilon_r=15,
)
topology = topfile.topology
system = topfile.create_system(nonbonded_cutoff=1.1 * unit.nanometer)
barostat = mm.MonteCarloBarostat(pressure, temperature)
system.addForce(barostat)

10

## Add Chemical Shift restraints

In [5]:
napshift_force = get_napshift_force(topology, 'Data/2ND3/CS.txt', model_type='martini')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

/home/mina/anaconda3/envs/build_openmmnapshift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/build_openmmnapshift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/build_openmmnapshift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/build

11

## Create the OpenMM simulation

In [6]:
integrator = mm.LangevinMiddleIntegrator(temperature, 0.01/unit.picosecond, timestep)
platform = mm.Platform.getPlatformByName('CUDA')
simulation = app.Simulation(topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(conf.getPositions())
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)


## Add reporters

In [7]:
xtc_reporter = app.XTCReporter('Data/2ND3/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

## Simulate!

In [8]:
print(f"Simulating without CS restraints")
simulation.step(100000)

warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

Simulating without CS restraints
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Speed (ns/day)"
1000,9.999999999999831,-48877.9801527012,3524.4760565582014,-45353.504096143,176.69738875112753,166.5723472024473,0
2000,20.000000000000327,-48709.043160574234,3804.639717124681,-44904.403443449555,190.74321753550493,167.55359456910114,2.81e+03
3000,30.00000000000189,-48515.37766130662,3930.8745731177823,-44584.50308818884,197.07192258709716,167.43662675416553,2.83e+03
4000,40.00000000000061,-48285.62034921634,4197.9689543844015,-44087.65139483194,210.46253127972577,168.45286336029275,2.82e+03
5000,49.99999999999862,-48030.45782359078,4377.333851128017,-43653.12397246277,219.45487748370658,169.32312212539526,2.81e+03


KeyboardInterrupt: 